# Rolling-Shutter Correction: Complete Proposed-Method Notebook

**Revised project plan:** implement the method from the research-paper draft and the supplied architecture diagram, but begin with a larger CNN targeting approximately **1 million trainable parameters** rather than the paper draft's approximately **0.25M** design. Train and evaluate this larger baseline first; investigate optimization afterward.

This notebook consolidates the conversation:
- The research paper: *A Lightweight CNN-Based Approach to Real-Time Rolling Shutter Correction on Resource-Constrained Devices*.
- The requested architectural change from ~0.25M to ~1M parameters.
- The diagram: low-resolution flow prediction, flow upsampling/rescaling, full-resolution warping, pixel + perceptual loss, and backpropagation.
- The earlier `train_flownet.py` concept and the revised `train_rsc.py` implementation.

**Research integrity note:** the paper draft says its proposed CNN has not yet been trained. The reported PSNR/SSIM/latency/quantization experiments in the draft use pretrained RS-Diffusion as a testbed. Those are not results of the proposed CNN and must not be presented as this notebook's results.

## 1. Problem and objective

A rolling-shutter camera reads sensor rows at different times. During camera or scene motion, different rows capture different moments, causing skew, wobble, and curved/jelly-like distortion.

**Goal:** from one distorted rolling-shutter (RS) image, predict a dense 2D correction flow and warp the original full-resolution image toward a global-shutter (GS) target.

### Revised objective
1. Accept a full-resolution RS image.
2. Downsample it for CNN processing.
3. Predict a dense flow field using a coarse-to-fine CNN encoder-decoder.
4. Upsample and rescale the flow to full-resolution pixel units.
5. Warp the original full-resolution RS image.
6. Train using pixel reconstruction and perceptual losses against the GS image.

### Requested change from the paper
| Detail | Paper draft | Revised baseline |
|---|---:|---:|
| Parameters | ~0.25M | ~1M target |
| Encoder channels | 24, 32, 48, 64 | 48, 64, 96, 128 |
| CNN input / flow grid | 64×64 | 64×64 |
| Full-resolution example | 256×256 | 256×256 |
| Flow estimation | Coarse-to-fine | Coarse-to-fine |
| Warp | Full-resolution | Full-resolution |
| Backbone | Plain CNN | Plain CNN |
| Transformer/diffusion in proposed CNN | None | None |

The ~1M value is a target, not a guaranteed count. The notebook prints the exact parameter count so the architecture can be adjusted if necessary.

## 2. Architecture diagram explained

```text
Captured RS image (full resolution)
       |-------------------------------+
       |                               |
       v                               |
Downsample to 64×64                    |
       |                               |
       v                               |
CNN convolutional encoder-decoder      |
       |                               |
       v                               |
Low-resolution flow (dx, dy)           |
       |                               |
       v                               |
Upsample + rescale flow                |
       |                               |
       +-------------------------------+
                       |
                       v
              Full-resolution warp
                       |
                       v
                Corrected image
                       |
                       v
          Pixel + perceptual loss vs GS
                       |
                       v
             Backpropagation to CNN
```

**Blocks:** the CNN predicts the flow; the upsampling/rescaling converts low-grid displacements into original-image pixel units; the warp samples the original RS image at displaced coordinates; the loss compares the corrected image with GS ground truth. The dashed gradient arrows in the supplied diagram represent backpropagation through the differentiable warp and flow resizing.

At inference time, the GS target and loss are not needed.

## 3. Method equations and important details

### 3.1 Low-resolution flow prediction
For full-resolution image \(I\) of size \(H\times W\), create a low-resolution image \(I_{lr}\) of size \(h\times w\). The CNN predicts a two-channel flow \(F_{lr}\), containing horizontal `dx` and vertical `dy` displacements.

For a 256×256 input and factor-four downsampling, the CNN processes 64×64.

### 3.2 Flow upsampling and value rescaling
Resizing the flow tensor alone is not enough because flow values are expressed in pixels of the low-resolution grid:

\[
F(p)=\left(\frac{W}{w}U(F_{lr})_x(p),\frac{H}{h}U(F_{lr})_y(p)\right)
\]

Horizontal and vertical flow components are scaled independently. Omitting this step causes incorrect displacement magnitudes.

### 3.3 Full-resolution backward warp
\[
\hat I(p)=I(p+F(p))
\]
The implementation uses `torch.nn.functional.grid_sample`, which supports differentiable backward warping.

### 3.4 Training objective
\[
L=L_{pix}(\hat I,I_{gs})+\lambda L_{perc}(\hat I,I_{gs})
\]
The draft specifies pixel L1 loss plus perceptual feature L1 loss, with planned \(\lambda=0.1\). This notebook uses frozen pretrained VGG16 features as the perceptual representation.

## 4. Dataset requirements

The loader expects `.npz` files with:
| Key | Expected shape | Meaning |
|---|---|---|
| `condition` | H×W×3 | Distorted RS image |
| `gs` | H×W×3 | Global-shutter ground truth |
| `out_flow` | optional 2×H×W or H×W×2 | Ground-truth flow; not used by the current pixel + perceptual training objective |

Example:
```text
RS-Real/
└── train/
    ├── sample_0001.npz
    ├── sample_0002.npz
    └── ...
```
Do not assume the downloaded dataset has this exact layout. Inspect its files and key names first; adapt the loader if necessary.

## 5. Install dependencies
Run this in Jupyter or Colab. Select a GPU runtime if available.

In [ ]:
%pip install -q torch torchvision numpy pillow tqdm scikit-image matplotlib

## 6. Imports, device, and reproducibility

In [ ]:
import os
import argparse
import math
import time
import random
from pathlib import Path

import numpy as np
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from torchvision.models import vgg16, VGG16_Weights

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 7. Configure dataset paths and hyperparameters
**Change `DATA_DIR` to the folder containing your training `.npz` files.** Start with a small batch size because full-resolution warping and VGG perceptual loss use GPU memory.

In [ ]:
DATA_DIR = "D:/RS-Real/train"  # Pointing to downloaded RS-Real dataset
SAVE_DIR = "./checkpoints"

FLOW_SIZE = 64
IMAGE_SIZE = 256
BATCH_SIZE = 4
EPOCHS = 10
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
PERCEPTUAL_WEIGHT = 0.1
NUM_WORKERS = 0

os.makedirs(SAVE_DIR, exist_ok=True)
print("Dataset:", DATA_DIR)
print("Checkpoints:", SAVE_DIR)


## 8. Inspect the dataset
Confirm files exist and verify the NPZ keys and shapes before training.

In [ ]:
npz_files = sorted(Path(DATA_DIR).rglob("*.npz"))
print("Number of NPZ files:", len(npz_files))
if not npz_files:
    raise FileNotFoundError(f"No .npz files found under {DATA_DIR}")

with np.load(npz_files[0]) as sample:
    print("Example:", npz_files[0].name)
    print("Keys:", list(sample.keys()))
    for key in sample.keys():
        print(key, sample[key].shape, sample[key].dtype)
    missing = {"condition", "gs"} - set(sample.keys())

if missing:
    raise KeyError(f"Missing required keys: {missing}")

## 9. Dataset loader

The code below is extracted from the generated `train_rsc.py` implementation. Review the comments and execute this cell after the preceding setup cells.

In [ ]:
class RollingShutterDataset(Dataset):
    """
    Expects NPZ files containing paired arrays:
      condition: distorted RS image, HxWx3 RGB (uint8 [0,255] or float [0,1])
      gs:        GS target, same spatial size and RGB channel order
    Optional out_flow is not used by training; its format must be verified before EPE.
    """
    def __init__(self, root, flow_size=64, image_size=256):
        self.files = sorted(Path(root).rglob("*.npz"))
        if not self.files:
            raise FileNotFoundError(
                f"No .npz files found under {root!r}. Check DATA_DIR and the dataset format."
            )
        self.flow_size, self.image_size = int(flow_size), int(image_size)
        self.valid_files, self.bad_files = [], []
        for path in self.files:
            try:
                with np.load(path, allow_pickle=False) as d:
                    if not {"condition", "gs"}.issubset(d.files):
                        raise ValueError(f"missing condition/gs keys; found {d.files}")
                    rs, gs = np.asarray(d["condition"]), np.asarray(d["gs"])
                    if rs.ndim != 3 or gs.ndim != 3 or rs.shape[-1] != 3 or gs.shape[-1] != 3:
                        raise ValueError(f"expected HxWx3 arrays, got {rs.shape} and {gs.shape}")
                    if rs.shape != gs.shape:
                        raise ValueError(f"paired image shapes differ: {rs.shape} vs {gs.shape}")
                    if not np.isfinite(rs).all() or not np.isfinite(gs).all():
                        raise ValueError("image contains NaN or infinity")
                self.valid_files.append(path)
            except Exception as exc:
                self.bad_files.append((str(path), str(exc)))
        if not self.valid_files:
            raise ValueError(f"No valid paired NPZ samples found in {root!r}. First errors: {self.bad_files[:3]}")
        if self.bad_files:
            print(f"Warning: skipped {len(self.bad_files)} invalid files.")
            for item in self.bad_files[:5]:
                print(" ", item)

    def __len__(self):
        return len(self.valid_files)

    @staticmethod
    def _to_float_chw(array):
        arr = np.asarray(array)
        tensor = torch.from_numpy(arr.copy()).permute(2, 0, 1).float()
        if np.issubdtype(arr.dtype, np.integer) or (tensor.numel() and tensor.max().item() > 1.0):
            tensor = tensor / 255.0
        if tensor.min().item() < 0 or tensor.max().item() > 1.0:
            raise ValueError("Image values must be in [0,1] or [0,255].")
        return tensor

    def __getitem__(self, index):
        path = self.valid_files[index]
        with np.load(path, allow_pickle=False) as data:
            rs = self._to_float_chw(data["condition"])
            gs = self._to_float_chw(data["gs"])
        rs_full = F.interpolate(rs[None], size=(self.image_size, self.image_size),
                                 mode="bilinear", align_corners=False)[0]
        gs_full = F.interpolate(gs[None], size=(self.image_size, self.image_size),
                                 mode="bilinear", align_corners=False)[0]
        rs_low = F.interpolate(rs_full[None], size=(self.flow_size, self.flow_size),
                               mode="bilinear", align_corners=False)[0]
        return rs_low, rs_full, gs_full


## 10. CNN building blocks and coarse-to-fine network

The code below is extracted from the generated `train_rsc.py` implementation. Review the comments and execute this cell after the preceding setup cells.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class FlowHead(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, in_ch // 2, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_ch // 2, 2, 3, padding=1),
        )

    def forward(self, x):
        return self.net(x)


class CoarseToFineFlowNet(nn.Module):
    """
    Plain CNN encoder-decoder. Flow is predicted at 1/8, 1/4, 1/2,
    and full working resolution (64x64 by default).
    """
    def __init__(self, channels=(48, 64, 96, 128)):
        super().__init__()
        c1, c2, c3, c4 = channels

        self.enc1 = ConvBlock(3, c1, stride=1)       # 64x64
        self.enc2 = ConvBlock(c1, c2, stride=2)     # 32x32
        self.enc3 = ConvBlock(c2, c3, stride=2)     # 16x16
        self.enc4 = ConvBlock(c3, c4, stride=2)     # 8x8

        self.flow8 = FlowHead(c4)

        # Decoder at 16, 32 and 64 pixels.
        self.dec3 = ConvBlock(c4 + c3 + 2, c3)
        self.flow16 = FlowHead(c3)

        self.dec2 = ConvBlock(c3 + c2 + 2, c2)
        self.flow32 = FlowHead(c2)

        self.dec1 = ConvBlock(c2 + c1 + 2, c1)
        self.flow64 = FlowHead(c1)

    @staticmethod
    def upsample_flow(flow, size):
        # Flow values are in pixels of the current grid. Doubling the
        # grid dimensions therefore requires doubling displacement values.
        old_h, old_w = flow.shape[-2:]
        new_h, new_w = size
        up = F.interpolate(flow, size=size, mode="bilinear", align_corners=False)
        scale = up.new_tensor([new_w / old_w, new_h / old_h]).view(1, 2, 1, 1)
        return up * scale

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)

        flow = self.flow8(e4)

        flow = self.upsample_flow(flow, e3.shape[-2:])
        d3 = self.dec3(torch.cat([e3, F.interpolate(e4, size=e3.shape[-2:],
                                                    mode="bilinear", align_corners=False),
                                  flow], dim=1))
        flow = flow + self.flow16(d3)

        flow = self.upsample_flow(flow, e2.shape[-2:])
        d2 = self.dec2(torch.cat([e2, F.interpolate(d3, size=e2.shape[-2:],
                                                    mode="bilinear", align_corners=False),
                                  flow], dim=1))
        flow = flow + self.flow32(d2)

        flow = self.upsample_flow(flow, e1.shape[-2:])
        d1 = self.dec1(torch.cat([e1, F.interpolate(d2, size=e1.shape[-2:],
                                                    mode="bilinear", align_corners=False),
                                  flow], dim=1))
        flow = flow + self.flow64(d1)
        return flow


# ----------------------------- Full-res warp -------------------------------

## 11. Flow upsampling and rescaling

The code below is extracted from the generated `train_rsc.py` implementation. Review the comments and execute this cell after the preceding setup cells.

In [ ]:
def upsample_and_rescale_flow(flow_low, output_size):
    """Resize low-resolution flow and scale dx/dy into full-resolution pixels."""
    out_h, out_w = output_size
    low_h, low_w = flow_low.shape[-2:]
    flow = F.interpolate(flow_low, size=(out_h, out_w),
                         mode="bilinear", align_corners=False)
    scale = flow.new_tensor([out_w / low_w, out_h / low_h]).view(1, 2, 1, 1)
    return flow * scale

## 12. Differentiable full-resolution warp

The code below is extracted from the generated `train_rsc.py` implementation. Review the comments and execute this cell after the preceding setup cells.

In [ ]:
def warp_image(image, flow):
    """
    Backward warp: output(p) = input(p + flow(p)).
    image: Bx3xHxW, flow: Bx2xHxW in pixel units.
    """
    b, _, h, w = image.shape
    yy, xx = torch.meshgrid(
        torch.arange(h, device=image.device, dtype=image.dtype),
        torch.arange(w, device=image.device, dtype=image.dtype),
        indexing="ij"
    )
    base_x = xx.unsqueeze(0).expand(b, -1, -1)
    base_y = yy.unsqueeze(0).expand(b, -1, -1)

    sample_x = base_x + flow[:, 0]
    sample_y = base_y + flow[:, 1]

    # Convert pixel coordinates to [-1, 1] for grid_sample.
    norm_x = 2.0 * sample_x / max(w - 1, 1) - 1.0
    norm_y = 2.0 * sample_y / max(h - 1, 1) - 1.0
    grid = torch.stack((norm_x, norm_y), dim=-1)

    return F.grid_sample(
        image, grid, mode="bilinear",
        padding_mode="border", align_corners=True
    )


# ----------------------------- Losses --------------------------------------

## 13. Perceptual loss

The code below is extracted from the generated `train_rsc.py` implementation. Review the comments and execute this cell after the preceding setup cells.

In [ ]:
class VGGPerceptualLoss(nn.Module):
    """VGG16 feature L1 loss. Pretrained weights download on first use."""
    def __init__(self):
        super().__init__()
        weights = VGG16_Weights.IMAGENET1K_V1
        features = vgg16(weights=weights).features[:16]
        self.features = features.eval()
        for p in self.features.parameters():
            p.requires_grad = False

        self.register_buffer(
            "mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        )
        self.register_buffer(
            "std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        )

    def forward(self, prediction, target):
        pred = (prediction - self.mean) / self.std
        tgt = (target - self.mean) / self.std
        pred_features = self.features(pred)
        with torch.no_grad():
            target_features = self.features(tgt)
        return F.l1_loss(pred_features, target_features)


# ----------------------------- Training ------------------------------------

## 14. Parameter count and model-shape test

The code below is extracted from the generated `train_rsc.py` implementation. Review the comments and execute this cell after the preceding setup cells.

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 15. Training loop, validation, and checkpoint saving

The implementation below trains only on the training subset, evaluates on a separate validation subset after every epoch, and saves the best checkpoint by validation loss plus the latest checkpoint. The held-out test subset is used only after training.

In [ ]:
def train_model(model, train_dataset, val_dataset, *, epochs=60, batch_size=4,
                learning_rate=2e-4, weight_decay=1e-4, perceptual_weight=0.1,
                workers=2, save_dir="./rsc_checkpoints", grad_clip=1.0):
    """Train on train_dataset only; evaluate each epoch on a disjoint validation subset."""
    device = next(model.parameters()).device
    generator = torch.Generator().manual_seed(RANDOM_SEED)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                              num_workers=workers, pin_memory=device.type == "cuda",
                              generator=generator)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                            num_workers=workers, pin_memory=device.type == "cuda")
    perceptual_loss = VGGPerceptualLoss().to(device).eval()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    os.makedirs(save_dir, exist_ok=True)
    best_val = float("inf")
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_total = train_pixel = train_perc = 0.0
        for rs_low, rs_full, gs_full in tqdm(train_loader, desc=f"Train {epoch}/{epochs}"):
            rs_low, rs_full, gs_full = [x.to(device, non_blocking=True) for x in (rs_low, rs_full, gs_full)]
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                flow_low = model(rs_low)
                flow_full = upsample_and_rescale_flow(flow_low, rs_full.shape[-2:])
                corrected = warp_image(rs_full, flow_full)
                pixel = F.l1_loss(corrected, gs_full)
                perceptual = perceptual_loss(corrected.float(), gs_full.float())
                loss = pixel + perceptual_weight * perceptual
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
            train_total += loss.item()
            train_pixel += pixel.item()
            train_perc += perceptual.item()

        model.eval()
        val_total = val_pixel = val_perc = 0.0
        with torch.no_grad():
            for rs_low, rs_full, gs_full in tqdm(val_loader, desc=f"Val {epoch}/{epochs}"):
                rs_low, rs_full, gs_full = [x.to(device, non_blocking=True) for x in (rs_low, rs_full, gs_full)]
                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                    flow_low = model(rs_low)
                    flow_full = upsample_and_rescale_flow(flow_low, rs_full.shape[-2:])
                    corrected = warp_image(rs_full, flow_full)
                    pixel = F.l1_loss(corrected, gs_full)
                    perceptual = perceptual_loss(corrected.float(), gs_full.float())
                    loss = pixel + perceptual_weight * perceptual
                val_total += loss.item()
                val_pixel += pixel.item()
                val_perc += perceptual.item()

        scheduler.step()
        row = {
            "epoch": epoch,
            "loss": train_total / max(len(train_loader), 1),
            "pixel_loss": train_pixel / max(len(train_loader), 1),
            "perceptual_loss": train_perc / max(len(train_loader), 1),
            "val_loss": val_total / max(len(val_loader), 1),
            "val_pixel_loss": val_pixel / max(len(val_loader), 1),
            "val_perceptual_loss": val_perc / max(len(val_loader), 1),
            "lr": optimizer.param_groups[0]["lr"],
        }
        history.append(row)
        print(row)
        checkpoint = {"epoch": epoch, "model_state_dict": model.state_dict(),
                      "optimizer_state_dict": optimizer.state_dict(), "history": history,
                      "trainable_parameters": count_parameters(model), "metrics": row}
        torch.save(checkpoint, os.path.join(save_dir, "last_model.pt"))
        if row["val_loss"] < best_val:
            best_val = row["val_loss"]
            torch.save(checkpoint, os.path.join(save_dir, "best_model.pt"))
            print("Saved best validation checkpoint.")
    return history


## 16. Command-line entry point

The code below is extracted from the generated `train_rsc.py` implementation. Review the comments and execute this cell after the preceding setup cells.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Train the rolling-shutter correction CNN.")
    parser.add_argument("--data_dir", required=True)
    parser.add_argument("--save_dir", default="./rsc_checkpoints")
    parser.add_argument("--flow_size", type=int, default=64)
    parser.add_argument("--image_size", type=int, default=256)
    parser.add_argument("--batch_size", type=int, default=4)
    parser.add_argument("--epochs", type=int, default=60)
    parser.add_argument("--lr", type=float, default=2e-4)
    parser.add_argument("--weight_decay", type=float, default=1e-4)
    parser.add_argument("--perceptual_weight", type=float, default=0.1)
    parser.add_argument("--grad_clip", type=float, default=1.0)
    parser.add_argument("--workers", type=int, default=2)
    args = parser.parse_args()
    dataset = RollingShutterDataset(args.data_dir, args.flow_size, args.image_size)
    if len(dataset) < 3:
        raise ValueError("At least 3 valid paired samples are required for train/validation/test splits.")
    n = len(dataset)
    n_train, n_val = max(1, int(0.70*n)), max(1, int(0.15*n))
    n_test = n - n_train - n_val
    if n_test < 1:
        n_train, n_val, n_test = n-2, 1, 1
    train_ds, val_ds, _test_ds = torch.utils.data.random_split(
        dataset, [n_train, n_val, n_test],
        generator=torch.Generator().manual_seed(RANDOM_SEED))
    model = CoarseToFineFlowNet().to(DEVICE)
    print(f"Trainable parameters: {count_parameters(model):,}")
    train_model(model, train_ds, val_ds, epochs=args.epochs, batch_size=args.batch_size,
                learning_rate=args.lr, weight_decay=args.weight_decay,
                perceptual_weight=args.perceptual_weight, workers=args.workers,
                save_dir=args.save_dir, grad_clip=args.grad_clip)

if __name__ == "__main__" and False:
    # Set to True only when running as a standalone Python script, not in a notebook.
    main()


## 17. Create dataset and model; verify tensor shapes and exact parameter count

Run this after the class/function cells above. The model uses channels `(48, 64, 96, 128)`. The exact count is printed rather than assumed to be exactly 1M.

In [ ]:
# Load and validate samples, then create disjoint train/validation/test splits.
dataset = RollingShutterDataset(DATA_DIR, flow_size=FLOW_SIZE, image_size=IMAGE_SIZE)
print("Valid dataset samples:", len(dataset))
if len(dataset) < 3:
    raise ValueError("Need at least 3 valid paired samples to create train/validation/test splits.")

n = len(dataset)
n_train = max(1, int(0.70 * n))
n_val = max(1, int(0.15 * n))
n_test = n - n_train - n_val
if n_test < 1:  # handles very small datasets without overlapping splits
    n_train, n_val, n_test = n - 2, 1, 1

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(RANDOM_SEED)
)
print(f"Split sizes — train: {len(train_dataset)}, validation: {len(val_dataset)}, test: {len(test_dataset)}")

rs_low_ex, rs_full_ex, gs_ex = dataset[0]
print("Low-resolution RS:", tuple(rs_low_ex.shape))
print("Full-resolution RS:", tuple(rs_full_ex.shape))
print("Full-resolution GS:", tuple(gs_ex.shape))

model = CoarseToFineFlowNet().to(DEVICE)
params = count_parameters(model)
print(f"Actual trainable parameters: {params:,} ({params / 1_000_000:.3f}M)")
if not (700_000 <= params <= 1_300_000):
    print("WARNING: parameter count is outside the broad ~1M target band; inspect architecture.")
dummy = torch.randn(2, 3, FLOW_SIZE, FLOW_SIZE, device=DEVICE)
with torch.no_grad():
    dummy_flow = model(dummy)
print("Input:", tuple(dummy.shape))
print("Predicted flow:", tuple(dummy_flow.shape))
assert dummy_flow.shape == (2, 2, FLOW_SIZE, FLOW_SIZE), "Expected Bx2xFLOW_SIZExFLOW_SIZE flow."


## 18. Flow-rescaling sanity check

A constant one-pixel displacement on a 64×64 grid should become a four-pixel displacement when applied to a 256×256 image. This checks that the flow values are rescaled, not merely interpolated.

In [ ]:
test_flow = torch.ones(1, 2, 64, 64)
test_full_flow = upsample_and_rescale_flow(test_flow, (256, 256))
print("Expected dx=4, actual:", test_full_flow[0, 0, 100, 100].item())
print("Expected dy=4, actual:", test_full_flow[0, 1, 100, 100].item())

## 19. Identity-warp sanity check

When the flow is all zeros, the warped output should closely match the input. A small floating-point error is normal.

In [ ]:
test_image = torch.rand(1, 3, 32, 32, device=DEVICE)
zero_flow = torch.zeros(1, 2, 32, 32, device=DEVICE)
identity_result = warp_image(test_image, zero_flow)
print("Maximum identity-warp error:",
      (test_image - identity_result).abs().max().item())

## 20. Complete forward pipeline

This cell explicitly mirrors the architecture diagram: low-res RS → CNN flow → full-res flow → warp original RS → corrected output.

In [ ]:
def run_model_pipeline(model, rs_low, rs_full):
    flow_low = model(rs_low)
    flow_full = upsample_and_rescale_flow(flow_low, rs_full.shape[-2:])
    corrected = warp_image(rs_full, flow_full)
    return corrected, flow_low, flow_full

with torch.no_grad():
    demo_corrected, demo_flow_low, demo_flow_full = run_model_pipeline(
        model,
        torch.rand(2, 3, FLOW_SIZE, FLOW_SIZE, device=DEVICE),
        torch.rand(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
    )
print("Low flow:", tuple(demo_flow_low.shape))
print("Full flow:", tuple(demo_flow_full.shape))
print("Corrected image:", tuple(demo_corrected.shape))

## 21. Smoke test and full training

Run the smoke test first by setting `RUN_SMOKE_TEST = True`. Confirm that loss/backpropagation and checkpoint saving work. Then set `RUN_FULL_TRAINING = True` for the planned 60-epoch baseline. Both flags default to `False` so opening the notebook does not accidentally start a long training job. No training is claimed until you run it.

In [ ]:
# Run a smoke test first. This is a real training run and may download VGG16 weights.
# Keep epochs=1 and batch_size=1 to verify forward/backward/checkpointing.
RUN_SMOKE_TEST = False
RUN_FULL_TRAINING = False

if RUN_SMOKE_TEST:
    smoke_model = CoarseToFineFlowNet().to(DEVICE)
    smoke_history = train_model(
        smoke_model, train_dataset, val_dataset,
        epochs=1, batch_size=1, learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY, perceptual_weight=PERCEPTUAL_WEIGHT,
        workers=0, save_dir=os.path.join(SAVE_DIR, "smoke_test")
    )

if RUN_FULL_TRAINING:
    history = train_model(
        model, train_dataset, val_dataset,
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
        perceptual_weight=PERCEPTUAL_WEIGHT, workers=NUM_WORKERS,
        save_dir=SAVE_DIR
    )


## 22. Training curves

Run after training if the `history` list has been returned. It plots total, pixel, and perceptual loss.

In [ ]:
def plot_training_history(history):
    if not history:
        raise ValueError("No history supplied. Run training first.")
    epochs = [row["epoch"] for row in history]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(epochs, [row["loss"] for row in history], label="Train total")
    axes[0].plot(epochs, [row["val_loss"] for row in history], label="Validation total")
    axes[0].set_title("Total loss"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].grid(True); axes[0].legend()
    axes[1].plot(epochs, [row["pixel_loss"] for row in history], label="Train pixel L1")
    axes[1].plot(epochs, [row["val_pixel_loss"] for row in history], label="Validation pixel L1")
    axes[1].plot(epochs, [row["perceptual_loss"] for row in history], label="Train perceptual")
    axes[1].plot(epochs, [row["val_perceptual_loss"] for row in history], label="Validation perceptual")
    axes[1].set_title("Loss components"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
    axes[1].grid(True); axes[1].legend()
    plt.tight_layout(); plt.show()

# After training: plot_training_history(history)


## 23. Load a checkpoint and visualize input / output / ground truth

For a scientifically valid result, visualize samples from a held-out test split. The example below is a simple loader demonstration using the dataset object.

In [ ]:
def load_checkpoint(model, checkpoint_path, device=DEVICE):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    print("Loaded epoch:", checkpoint.get("epoch", "unknown"))
    return checkpoint


def show_correction(model, dataset, index=0):
    rs_low, rs_full, gs_full = dataset[index]
    with torch.no_grad():
        corrected, _, _ = run_model_pipeline(
            model, rs_low.unsqueeze(0).to(DEVICE), rs_full.unsqueeze(0).to(DEVICE)
        )

    images = [
        rs_full.permute(1, 2, 0).cpu().numpy(),
        corrected[0].permute(1, 2, 0).cpu().numpy(),
        gs_full.permute(1, 2, 0).cpu().numpy()
    ]
    titles = ["Input RS", "Corrected output", "GS ground truth"]

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    for ax, image, title in zip(axes, images, titles):
        ax.imshow(np.clip(image, 0, 1))
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

# load_checkpoint(model, os.path.join(SAVE_DIR, "best_model.pt"))
# show_correction(model, dataset, index=0)

## 24. Evaluation metrics

The paper uses PSNR, SSIM, and EPE. PSNR and SSIM compare the corrected image with GS ground truth. EPE requires aligned ground-truth flow and consistent flow direction, axis order, and pixel units.

Do not report EPE unless you load and correctly transform the dataset's ground-truth `out_flow`.

In [ ]:
def calculate_psnr(prediction, target, data_range=1.0):
    mse = F.mse_loss(prediction, target).item()
    if mse == 0:
        return float("inf")
    return 10.0 * math.log10((data_range ** 2) / mse)


def calculate_ssim(prediction, target):
    from skimage.metrics import structural_similarity
    pred = prediction.detach().cpu().squeeze(0).permute(1, 2, 0).numpy()
    tgt = target.detach().cpu().squeeze(0).permute(1, 2, 0).numpy()
    return structural_similarity(tgt, pred, channel_axis=2, data_range=1.0)


def endpoint_error(predicted_flow, target_flow):
    # Both tensors must be [B, 2, H, W] in pixels at the same resolution.
    error = torch.sqrt(((predicted_flow - target_flow) ** 2).sum(dim=1) + 1e-12)
    return error.mean().item()

In [ ]:
def evaluate_dataset(model, test_dataset, batch_size=1, workers=0):
    """Evaluate corrected images on the held-out test split; EPE is omitted without verified GT flow."""
    device = next(model.parameters()).device
    loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=workers)
    model.eval()
    rows = []
    with torch.no_grad():
        for index, (rs_low, rs_full, gs_full) in enumerate(tqdm(loader, desc="Test evaluation")):
            rs_low, rs_full, gs_full = [x.to(device) for x in (rs_low, rs_full, gs_full)]
            corrected, flow_low, flow_full = run_model_pipeline(model, rs_low, rs_full)
            for j in range(corrected.shape[0]):
                rows.append({
                    "sample_index": index * batch_size + j,
                    "psnr_db": calculate_psnr(corrected[j:j+1], gs_full[j:j+1]),
                    "ssim": calculate_ssim(corrected[j:j+1], gs_full[j:j+1]),
                })
    if not rows:
        raise ValueError("Test split is empty.")
    print(f"Test samples: {len(rows)}")
    print(f"Mean PSNR: {np.mean([r['psnr_db'] for r in rows]):.3f} dB")
    print(f"Mean SSIM: {np.mean([r['ssim'] for r in rows]):.4f}")
    print(f"Trainable parameters: {count_parameters(model):,}")
    return rows

# After training, load the best checkpoint, then run:
# load_checkpoint(model, os.path.join(SAVE_DIR, "best_model.pt"))
# test_results = evaluate_dataset(model, test_dataset)


## 25. Inference latency

Latency must be measured after warm-up and with GPU synchronization. A Kaggle/Colab T4 measurement is not an embedded-device benchmark. Use batch size 1 for per-image latency.

In [ ]:
def measure_inference(model, rs_low, rs_full, warmup=5, repeats=30):
    model.eval()
    rs_low = rs_low.unsqueeze(0).to(DEVICE)
    rs_full = rs_full.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        for _ in range(warmup):
            run_model_pipeline(model, rs_low, rs_full)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

        times_ms = []
        for _ in range(repeats):
            start = time.perf_counter()
            run_model_pipeline(model, rs_low, rs_full)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            times_ms.append((time.perf_counter() - start) * 1000)

    median_ms = float(np.median(times_ms))
    return median_ms, 1000.0 / median_ms

## 26. Later optimization plan — keep this separate from the baseline

Do **not** optimize before recording the larger baseline. Later experiments can include:
1. Weight-only INT8 quantization.
2. Structured channel/filter pruning followed by fine-tuning.
3. Depthwise-separable convolutions as a separate architecture variant.
4. Knowledge distillation from the larger model to a smaller student.
5. ONNX/TensorRT or another target runtime for actual deployment measurement.

Simulated quantization or unstructured sparsity does not prove a real speedup unless the runtime exploits the compressed representation.

## 27. Relationship to the earlier `train_flownet.py`

The earlier script discussed in the conversation was a FlowNet-style implementation reported in the chat as approximately 969,644 parameters. It used low-resolution flow prediction, full-resolution warping, flow L1 plus a weighted image/photometric loss, AdamW, OneCycleLR, mixed precision, and PSNR evaluation.

This notebook follows the research-paper formulation more closely:
- explicit coarse-to-fine flow refinement;
- low-resolution flow prediction;
- explicit flow upsampling **and value rescaling**;
- full-resolution backward warping;
- pixel L1 + perceptual feature loss with weight 0.1;
- cosine learning-rate decay.

These implementations are not identical. Compare them experimentally rather than assuming equivalent behavior.

## 28. Paper findings for context — not results of this CNN

The uploaded paper draft reports a reference study using pretrained RS-Diffusion on RS-Real:
- At 256×256, full-resolution warping with rescaled flow: 26.16 dB PSNR / 0.860 SSIM at 8 DDIM steps.
- Low-resolution warp followed by upscaling: 21.47 dB / 0.615.
- Omitting flow value rescaling: 16.47 dB PSNR at 8 steps in that study.
- One DDIM step: 25.77 dB PSNR, 0.856 SSIM, 37.0 ms on a T4; eight steps: 26.16 dB, 0.860 SSIM, 186.0 ms.
- Weight-only simulated 8-bit quantization caused about 0.07 dB PSNR loss for RS-Diffusion.
- The paper says no embedded device was used and the proposed CNN had not yet been trained.

These values motivate the design. They must not be attributed to the CNN trained in this notebook.

## 29. Limitations and verification checklist

1. Verify the actual dataset folder, file format, and NPZ keys.
2. Create separate training, validation, and test splits; this notebook's loader only points to the supplied training directory.
3. Verify the printed parameter count and adjust channel widths if a closer 1M target is required.
4. Document the perceptual network, feature layer, and preprocessing.
5. Confirm RS and GS pairs are aligned and resized consistently.
6. Confirm flow direction, sign, axis order, and units before computing EPE.
7. Report test-set size, image resolution, seed, GPU, framework version, and latency protocol.
8. Do not describe a GPU benchmark as embedded-device performance.
9. Keep paper-reported RS-Diffusion results separate from your own CNN results.
10. Record baseline results before starting pruning or quantization.

## 30. Experiment results template

Fill this table only after running the corresponding experiments.

| Model / setting | Parameters | PSNR | SSIM | EPE | Latency (ms) | Device |
|---|---:|---:|---:|---:|---:|---|
| Uncorrected RS input | — | — | — | — | — | — |
| Proposed CNN (~1M target) | measured | — | — | — | — | — |
| CNN after optimization | measured | — | — | — | — | — |
| RS-Diffusion reference | 79.9M (paper) | paper value | paper value | paper value | paper value | T4 in paper |

The RS-Diffusion row is literature context, not a result produced by this notebook.

## 31. Final execution checklist

- [ ] Dataset path is correct.
- [ ] NPZ keys `condition` and `gs` exist.
- [ ] Image shapes and value ranges are verified.
- [ ] CNN forward pass succeeds.
- [ ] Exact trainable parameter count is recorded.
- [ ] Flow-rescaling sanity check gives expected scale.
- [ ] Zero-flow identity warp test has small error.
- [ ] A short smoke-test training run completes.
- [ ] Best checkpoint is saved.
- [ ] Separate validation and test splits are prepared.
- [ ] PSNR / SSIM / EPE use a documented protocol.
- [ ] Latency and memory are measured on a named device.
- [ ] Optimization is performed only after baseline evaluation.

**End of project notebook.**